# ==========================================
# 03_Feature_Engineering.ipynb
# Predictive Maintenance using CWRU Dataset
# MSc Data Science Dissertation
# ==========================================

# Import Libraries

In [34]:
import numpy as np
import pandas as pd

from scipy.io import loadmat
from scipy.stats import skew
from scipy.stats import kurtosis

from scipy.fft import fft
from scipy.fft import fftfreq

import pywt

# Define Bearing Frequencies

In [35]:
FS = 12000

BPFO = 107
BPFI = 162
BSF = 141

# Load Signal Function

In [36]:
def load_cwru_signal(filename, variable_name):

    data = loadmat(filename)

    signal = data[variable_name]

    signal = signal.flatten()

    return signal

# Window Function

In [37]:
def create_windows(
        signal,
        window_size=1024,
        step_size=512):

    windows = []

    for start in range(
            0,
            len(signal)-window_size,
            step_size):

        end = start + window_size

        windows.append(
            signal[start:end]
        )

    return np.array(windows)

# Time Domain Features

In [38]:
def time_features(window):

    mean = np.mean(window)

    rms = np.sqrt(np.mean(window**2))

    std = np.std(window)

    peak = np.max(np.abs(window))

    peak_to_peak = np.ptp(window)

    skewness = skew(window)

    kurt = kurtosis(window)

    crest_factor = peak / rms

    shape_factor = rms / np.mean(np.abs(window))

    impulse_factor = peak / np.mean(np.abs(window))

    return {
        "Mean": mean,
        "RMS": rms,
        "Std": std,
        "Peak": peak,
        "Peak_to_Peak": peak_to_peak,
        "Skewness": skewness,
        "Kurtosis": kurt,
        "Crest_Factor": crest_factor,
        "Shape_Factor": shape_factor,
        "Impulse_Factor": impulse_factor
    }

In [39]:
normal_signal = load_cwru_signal(
    '97.mat',
    'X097_DE_time'
)

print(normal_signal.shape)

(243938,)


# Validation of Time Domain Features

In [40]:
test_window = normal_signal[:1024]

features = time_features(test_window)

print(features)

{'Mean': np.float64(0.01242096814903846), 'RMS': np.float64(0.07598206978887359), 'Std': np.float64(0.07495995250560014), 'Peak': np.float64(0.21779446153846155), 'Peak_to_Peak': np.float64(0.41639630769230773), 'Skewness': np.float64(-0.16297611297662787), 'Kurtosis': np.float64(-0.24366426455323964), 'Crest_Factor': np.float64(2.8663928495713895), 'Shape_Factor': np.float64(1.2326922553049786), 'Impulse_Factor': np.float64(3.5333802663282206)}


# FFT Functions

In [47]:
def get_fft(window):

    N = len(window)

    fft_values = np.fft.fft(window)

    fft_values = np.abs(fft_values[:N//2])

    freqs = np.fft.fftfreq(
        N,
        d=1/FS
    )[:N//2]

    return freqs, fft_values

Frequency Features
Spectral Energy
Dominant Frequency

Amplitude BPFO
Amplitude BPFI
Amplitude BSF

BPFO/BPFI Ratio

Band Power Features

0-500 Hz

500-1000 Hz

1000-2000 Hz

# Amplitude at Frequency

In [48]:
def amplitude_at_frequency(freqs, spectrum, target_freq):

    idx = np.argmin(
        np.abs(freqs - target_freq)
    )

    return spectrum[idx]

# Band Power Function

In [49]:
def band_power(
    freqs,
    spectrum,
    low_freq,
    high_freq
):

    mask = (
        (freqs >= low_freq)
        &
        (freqs < high_freq)
    )

    return np.sum(
        spectrum[mask]
    )

# Wavelet Function

In [50]:
def wavelet_energies(window):

    coeffs = pywt.wavedec(
        window,
        'db4',
        level=4
    )

    d4 = coeffs[1]
    d3 = coeffs[2]
    d2 = coeffs[3]
    d1 = coeffs[4]

    e_d1 = np.sum(d1**2)
    e_d2 = np.sum(d2**2)
    e_d3 = np.sum(d3**2)
    e_d4 = np.sum(d4**2)

    return e_d1, e_d2, e_d3, e_d4

# Master Feature Extraction Function

In [51]:
def extract_features(window):

    time_feats = time_features(window)

    freqs, spectrum = get_fft(window)

    spectral_energy = np.sum(
        spectrum**2
    )

    dominant_frequency = freqs[
        np.argmax(spectrum)
    ]

    amp_bpfo = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFO
    )

    amp_bpfi = amplitude_at_frequency(
        freqs,
        spectrum,
        BPFI
    )

    amp_bsf = amplitude_at_frequency(
        freqs,
        spectrum,
        BSF
    )

    bpfo_bpfi_ratio = (
        amp_bpfo / amp_bpfi
        if amp_bpfi != 0
        else 0
    )

    band_0_500 = band_power(
        freqs,
        spectrum,
        0,
        500
    )

    band_500_1000 = band_power(
        freqs,
        spectrum,
        500,
        1000
    )

    band_1000_2000 = band_power(
        freqs,
        spectrum,
        1000,
        2000
    )

    e_d1, e_d2, e_d3, e_d4 = wavelet_energies(window)

    return {

        **time_feats,

        "Spectral_Energy": spectral_energy,

        "Dominant_Frequency": dominant_frequency,

        "Amp_BPFO": amp_bpfo,

        "Amp_BPFI": amp_bpfi,

        "Amp_BSF": amp_bsf,

        "BPFO_BPFI_Ratio": bpfo_bpfi_ratio,

        "Band_0_500": band_0_500,

        "Band_500_1000": band_500_1000,

        "Band_1000_2000": band_1000_2000,

        "Wavelet_D1": e_d1,

        "Wavelet_D2": e_d2,

        "Wavelet_D3": e_d3,

        "Wavelet_D4": e_d4
    }

# Test Master Feature Function

In [52]:
test_window = normal_signal[:1024]

features = extract_features(
    test_window
)

print(features)

{'Mean': np.float64(0.01242096814903846), 'RMS': np.float64(0.07598206978887359), 'Std': np.float64(0.07495995250560014), 'Peak': np.float64(0.21779446153846155), 'Peak_to_Peak': np.float64(0.41639630769230773), 'Skewness': np.float64(-0.16297611297662787), 'Kurtosis': np.float64(-0.24366426455323964), 'Crest_Factor': np.float64(2.8663928495713895), 'Shape_Factor': np.float64(1.2326922553049786), 'Impulse_Factor': np.float64(3.5333802663282206), 'Spectral_Energy': np.float64(3107.7448697119767), 'Dominant_Frequency': np.float64(1066.40625), 'Amp_BPFO': np.float64(6.685926632870019), 'Amp_BPFI': np.float64(13.120113715524916), 'Amp_BSF': np.float64(3.5453995199367787), 'BPFO_BPFI_Ratio': np.float64(0.5095936497073665), 'Band_0_500': np.float64(147.5511311431333), 'Band_500_1000': np.float64(95.6588459195146), 'Band_1000_2000': np.float64(161.24184903753869), 'Wavelet_D1': np.float64(0.03610324176120579), 'Wavelet_D2': np.float64(0.5597924884201901), 'Wavelet_D3': np.float64(3.4315397747

# File Mapping

In [ ]:
file_mapping = {

    # Normal
    97:0,
    98:0,
    99:0,
    100:0,

    # Inner Race
    105:1,
    106:1,
    107:1,
    108:1,

    # Ball
    118:2,
    119:2,
    120:2,
    121:2,

    # Outer
    130:3,
    131:3,
    132:3,
    133:3
}